In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split, cross_val_score

In [2]:
df = pd.read_csv(
    "kagglehub/datasets/nalisha/job-salary-prediction-dataset/versions/1/job_salary_prediction_dataset.csv" #ignore the path its huge ik
    )

FileNotFoundError: [Errno 2] No such file or directory: 'kagglehub/datasets/nalisha/job-salary-prediction-dataset/versions/1/job_salary_prediction_dataset.csv'

In [ ]:
df.dtypes

job_title             str
experience_years    int64
education_level       str
skills_count        int64
industry              str
company_size          str
location              str
remote_work           str
certifications      int64
salary              int64
dtype: object

In [ ]:
df["education_level"].unique()

<ArrowStringArray>
['Bachelor', 'PhD', 'High School', 'Diploma', 'Master']
Length: 5, dtype: str

In [ ]:
df["company_size"].unique()

<ArrowStringArray>
['Medium', 'Small', 'Large', 'Enterprise', 'Startup']
Length: 5, dtype: str

# **Numerical Features**
- No need to clean, no null values
- No need to power transform, very little/insignificant skewdness
- Standardize via RobustScaler().

# **Categorical Features**
- Ordinally encode education_level
- One-hot encode others

# **Model**
- Test with different models - linear regression, gradient boost regressor, etc...

In [ ]:
transformations = ColumnTransformer([
    ("numerical", RobustScaler(), make_column_selector(dtype_include="int64")),
    ("one-hot", OneHotEncoder(drop='first', sparse_output=False), ["job_title", "industry", "remote_work", "location", "company_size"]),
    ("education", OrdinalEncoder(categories=[["High School", "Diploma", "Bachelor", "Master", "PhD"]]), ["education_level"]),
])

In [ ]:
pipeline = Pipeline([
    ("transform", transformations),
    ("model", LinearRegression())
])

In [ ]:
x = df.drop(columns=["salary"])
y = df["salary"]

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    random_state = 11,
    test_size = 0.2
)

In [ ]:
models = [LinearRegression(), HistGradientBoostingRegressor()]

In [ ]:
for model in models:
    pipeline.set_params(model = model)
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    print("-" * 50)
    print(f"R2: {r2_score(y_test, y_pred)}, MAE: {mean_absolute_error(y_test, y_pred)}")


--------------------------------------------------
R2: 0.9603931472072851, MAE: 5708.912556084143
--------------------------------------------------
R2: 0.977039127436248, MAE: 4491.324826770334


In [ ]:
# look at histgradientboostingregressor's scores
print(pipeline.score(x_train, y_train))
print(pipeline.score(x_test, y_test))

0.9775607453060308
0.977039127436248


In [ ]:
cross_val_score(
    pipeline,
    x,
    y,
    cv = 5,
    scoring = "r2"
)

array([0.97733414, 0.97662531, 0.97789277, 0.9777769 , 0.97734328])

In [ ]:
import joblib
joblib.dump(pipeline, "salary_predictor.joblib")

['salary_predictor.joblib']